In [1]:
%matplotlib qt
from pathlib import Path
import pandas as pd
import numpy as np
import sys

from math import acos

sys.path.append(r'D:\Users\Csabi\dev\Cybathlon')

USE_ROLLING_WINDOW = False
ONLY_KONTROLL_DATA = False
FILTER_DATA = True  # 11 pix distance

# for windowing data
WINDOW_LEN = 100
WINDOW_STEP = 15

In [2]:
path = Path(r'D:\Users\Csabi\OneDrive - Pázmány Péter Katolikus Egyetem\1 Projects\Onkology')
file = path.joinpath('Összesített nyers eredmények', 'kiértékelés javított')
files = [file for file in file.rglob('*.xls') if '/DMSO/' not in str(file.as_posix()) and
         not str(file.as_posix()).endswith('/0528/A2058/1/044.xls') and
         not str(file.as_posix()).endswith('/0528/Mel JR Post/1/053.xls')]

In [3]:
from utils import calculate_init_data

def open_file(file, min_len=433):
    
    # df = pd.read_excel(file)._convert(numeric=True)
    
    dfs = pd.read_excel(file)
    # display(df)
    # df = df.astype(float)
    # display(df)

    # print(f'{file}\n{df.columns}\n\n')
    p = file.parent
    if len(p.name)==1:
        dfs['type'] = f'{p.name} uM'
    else:
        dfs['type'] = 'kontroll'
    p = p.parent
    dfs['cell line'] = p.name.removeprefix('Mel ')
    dfs.columns = [col.removeprefix(' ').removesuffix(' ') for col in dfs.columns]

    return calculate_init_data(dfs, min_len, filter_dist=FILTER_DATA)

# dfs = open_file(files[56])
# dfs[0]

In [4]:
from utils import get_features

labels = ['Cell ID unique', 'type', 'cell line']

cell_list = []
for file in files:
    cell_list.extend(open_file(file))

if USE_ROLLING_WINDOW:
    wl = WINDOW_LEN
    step = WINDOW_STEP
    win_cell_list = []
    for cdf in cell_list:
        win_cell_list.extend([w for w in cdf.rolling(window=wl, step=step) if len(w) == wl])
    clist = win_cell_list
else:
    clist = cell_list
    
df_ = pd.concat([get_features(w, labels) for w in clist], axis=1).T
df_

,total_dist,dist std,dist mean,max_disp,directionality,avg speed,msd,angle from last_mov [deg] -- std,angle from last_mov [deg] -- mav,angle from last_mov [deg] -- zc,angle from init [deg] -- std,diff angle from init [deg] -- std,diff angle from init [deg] -- mav,diff angle from init [deg] -- zc,Cell ID unique,type,cell line
0,1552.005961,12.493506,3.592606,550.138021,0.297348,0.359261,238.315834,0.0,0.0,2.0,40.834526,13.582218,1.085205,72.0,0,1 uM,A2058
1,2954.069092,16.607849,6.838123,554.637586,0.170457,0.683812,240.633322,0.0,0.0,2.0,96.197836,47.870355,9.87987,125.0,3,1 uM,A2058
2,3491.190024,19.109881,8.081458,499.909698,0.132667,0.808146,174.137192,0.0,0.0,2.0,30.267191,5.650669,1.614578,133.0,4,1 uM,A2058
3,1432.773708,11.643507,3.316606,285.105331,0.156835,0.331661,114.079129,0.0,0.0,2.0,80.227705,17.455075,2.446452,66.0,5,1 uM,A2058
4,2708.524032,18.480233,6.269732,245.876555,0.065684,0.626973,98.168256,0.0,0.0,2.0,87.76583,19.463112,3.853848,90.0,6,1 uM,A2058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3671,1773.403827,14.840244,4.105101,117.19401,0.040715,0.41051,67.972264,0.0,0.0,2.0,89.582729,34.106675,6.052095,68.0,4085,kontroll,WM983B
3672,181.737818,4.892592,0.420689,118.784997,0.42596,0.042069,17.678274,0.0,0.0,2.0,15.605693,9.970239,0.560451,8.0,4086,kontroll,WM983B
3673,489.147555,5.657191,1.132286,76.796669,0.131983,0.113229,32.198044,0.0,0.0,2.0,80.125892,23.848899,3.175773,33.0,4087,kontroll,WM983B
3674,1276.439286,12.916445,2.954721,203.032799,0.117782,0.295472,58.699956,0.0,0.0,2.0,56.689315,10.071547,1.024776,39.0,4088,kontroll,WM983B


In [5]:
file = path.joinpath('data', 'cell_line_profile.csv')
df_prof = pd.read_csv(file, sep=';')
df_prof = df_prof.drop('ID / plate', axis=1)
gr = ['cell line', 'type']
df_prof = df_prof.groupby(gr).mean() * 2.18  # (um)
# df_prof = df_prof.loc[df_[gr[0]].unique(),df_[gr[1]].unique(),:] 
df_prof

Area      Circ        AR     Round  Solidity
cell line type                                                         
A2058     1 uM      2895.178413  1.212703  6.976173  0.873557  1.881755
          2 uM      3645.282963  1.310624  6.554574  0.960169  1.952957
          kontroll  2953.229231  1.393590  5.878253  1.029094  1.964247
JL Post   1 uM      5292.694390  1.342614  4.268121  1.262512  1.876900
          2 uM      4600.158356  1.420106  4.570594  1.265236  1.893912
          kontroll  6146.417627  1.210048  4.805939  1.178974  1.754050
JR Post   1 uM      2808.415278  1.619377  4.691118  1.199091  2.065641
          2 uM      2808.255238  1.614999  4.634299  1.214260  2.038404
          kontroll  2210.607200  1.597853  4.391828  1.273905  1.995369
JR Pre    1 uM      3717.316765  1.655229  4.010110  1.327171  2.042468
          2 uM      3723.010606  1.694950  3.912968  1.335481  2.053494
          kontroll  3028.973750  1.610938  4.456574  1.225160  2.047674
WM983B    1 uM      2402.052958  1.621152  4.305408  1.291481  2.050643
          2 uM      2258.971268  1.643413  4.236354  1.237626  2.040357
          kontroll  2127.580909  1.521244  4.232866  1.252179  1.966955

In [6]:
# use only kontroll data
if ONLY_KONTROLL_DATA:
    gr = ['cell line']
    df_prof = df_prof.loc[(slice(None), 'kontroll'),:]
    display(df_prof)

In [7]:
df = pd.merge(df_, df_prof, on=gr, how='right')
profile = df[df_prof.columns]

In [8]:
labels = ['type', 'cell line', 'Cell ID unique'] 
x = df.drop(labels + list(profile.columns), axis=1)
y = df[labels]
y.drop_duplicates()

,type,cell line,Cell ID unique
0,1 uM,A2058,0
1,1 uM,A2058,3
2,1 uM,A2058,4
3,1 uM,A2058,5
4,1 uM,A2058,6
...,...,...,...
3671,kontroll,WM983B,4085
3672,kontroll,WM983B,4086
3673,kontroll,WM983B,4087
3674,kontroll,WM983B,4088


In [9]:
x

,total_dist,dist std,dist mean,max_disp,directionality,avg speed,msd,angle from last_mov [deg] -- std,angle from last_mov [deg] -- mav,angle from last_mov [deg] -- zc,angle from init [deg] -- std,diff angle from init [deg] -- std,diff angle from init [deg] -- mav,diff angle from init [deg] -- zc
0,1552.005961,12.493506,3.592606,550.138021,0.297348,0.359261,238.315834,0.0,0.0,2.0,40.834526,13.582218,1.085205,72.0
1,2954.069092,16.607849,6.838123,554.637586,0.170457,0.683812,240.633322,0.0,0.0,2.0,96.197836,47.870355,9.87987,125.0
2,3491.190024,19.109881,8.081458,499.909698,0.132667,0.808146,174.137192,0.0,0.0,2.0,30.267191,5.650669,1.614578,133.0
3,1432.773708,11.643507,3.316606,285.105331,0.156835,0.331661,114.079129,0.0,0.0,2.0,80.227705,17.455075,2.446452,66.0
4,2708.524032,18.480233,6.269732,245.876555,0.065684,0.626973,98.168256,0.0,0.0,2.0,87.76583,19.463112,3.853848,90.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3671,1773.403827,14.840244,4.105101,117.19401,0.040715,0.41051,67.972264,0.0,0.0,2.0,89.582729,34.106675,6.052095,68.0
3672,181.737818,4.892592,0.420689,118.784997,0.42596,0.042069,17.678274,0.0,0.0,2.0,15.605693,9.970239,0.560451,8.0
3673,489.147555,5.657191,1.132286,76.796669,0.131983,0.113229,32.198044,0.0,0.0,2.0,80.125892,23.848899,3.175773,33.0
3674,1276.439286,12.916445,2.954721,203.032799,0.117782,0.295472,58.699956,0.0,0.0,2.0,56.689315,10.071547,1.024776,39.0


In [10]:
f = x.copy()
dcols = x.columns

# normed with cell profile data
for col in dcols:
    for cell_par in profile.columns:
        f[f'{cell_par}_norm: {col}'] = f[col] / profile[cell_par]
            # break;
        # f[f'ContArea_norm: {tp}{col}'] = f[f'{tp}{col}'] / df_['cell size (um)'] * 2.18  # (um)

f

,total_dist,dist std,dist mean,max_disp,directionality,avg speed,msd,angle from last_mov [deg] -- std,angle from last_mov [deg] -- mav,angle from last_mov [deg] -- zc,...,Area_norm: diff angle from init [deg] -- mav,Circ_norm: diff angle from init [deg] -- mav,AR_norm: diff angle from init [deg] -- mav,Round_norm: diff angle from init [deg] -- mav,Solidity_norm: diff angle from init [deg] -- mav,Area_norm: diff angle from init [deg] -- zc,Circ_norm: diff angle from init [deg] -- zc,AR_norm: diff angle from init [deg] -- zc,Round_norm: diff angle from init [deg] -- zc,Solidity_norm: diff angle from init [deg] -- zc
0,1552.005961,12.493506,3.592606,550.138021,0.297348,0.359261,238.315834,0.0,0.0,2.0,...,0.000375,0.894864,0.155559,1.242282,0.576698,0.024869,59.37151,10.320845,82.421626,38.262149
1,2954.069092,16.607849,6.838123,554.637586,0.170457,0.683812,240.633322,0.0,0.0,2.0,...,0.003413,8.146984,1.416231,11.30993,5.250348,0.043175,103.075538,17.918134,143.0931,66.427343
2,3491.190024,19.109881,8.081458,499.909698,0.132667,0.808146,174.137192,0.0,0.0,2.0,...,0.000558,1.331388,0.231442,1.848279,0.858017,0.045938,109.672373,19.064894,152.251059,70.678693
3,1432.773708,11.643507,3.316606,285.105331,0.156835,0.331661,114.079129,0.0,0.0,2.0,...,0.000845,2.017355,0.350687,2.800563,1.30009,0.022797,54.423884,9.460775,75.553157,35.073637
4,2708.524032,18.480233,6.269732,245.876555,0.065684,0.626973,98.168256,0.0,0.0,2.0,...,0.001331,3.1779,0.55243,4.411672,2.048007,0.031086,74.214388,12.901056,103.027032,47.827687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3671,1773.403827,14.840244,4.105101,117.19401,0.040715,0.41051,67.972264,0.0,0.0,2.0,...,0.002845,3.978386,1.429786,4.833251,3.076886,0.031961,44.700269,16.064764,54.305344,34.571211
3672,181.737818,4.892592,0.420689,118.784997,0.42596,0.042069,17.678274,0.0,0.0,2.0,...,0.000263,0.368416,0.132405,0.447581,0.284933,0.00376,5.258855,1.889972,6.388864,4.067201
3673,489.147555,5.657191,1.132286,76.796669,0.131983,0.113229,32.198044,0.0,0.0,2.0,...,0.001493,2.087616,0.750265,2.536198,1.614564,0.015511,21.692778,7.796136,26.354064,16.777205
3674,1276.439286,12.916445,2.954721,203.032799,0.117782,0.295472,58.699956,0.0,0.0,2.0,...,0.000482,0.673643,0.2421,0.818394,0.520996,0.018331,25.636919,9.213615,31.145712,19.827606


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
# X = x
X = f
# X = f.drop(x.columns, axis=1)
X = X.fillna(0)

VERBOSE = False

REPEAT_CV = 10
selection = 15  # first n feature

# cv = StratifiedKFold(n_splits=5)
cv = StratifiedShuffleSplit(n_splits=5)
pipe = RandomForestClassifier(n_estimators=500, n_jobs=-2)
# pipe = make_pipeline(
#         # PCA(n_components=.97),
#         StandardScaler(),
#         RandomForestClassifier(n_estimators=500, n_jobs=-2)
# )
# pipe = get_ensemble_clf('voting', 'hard')

# print(X.isnull().values.any())
# with pd.option_context('display.max_rows', None):
#     display(X)
X

C:\Users\Csabi\AppData\Local\Temp\ipykernel_25108\4018361090.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.fillna(0)


,total_dist,dist std,dist mean,max_disp,directionality,avg speed,msd,angle from last_mov [deg] -- std,angle from last_mov [deg] -- mav,angle from last_mov [deg] -- zc,...,Area_norm: diff angle from init [deg] -- mav,Circ_norm: diff angle from init [deg] -- mav,AR_norm: diff angle from init [deg] -- mav,Round_norm: diff angle from init [deg] -- mav,Solidity_norm: diff angle from init [deg] -- mav,Area_norm: diff angle from init [deg] -- zc,Circ_norm: diff angle from init [deg] -- zc,AR_norm: diff angle from init [deg] -- zc,Round_norm: diff angle from init [deg] -- zc,Solidity_norm: diff angle from init [deg] -- zc
0,1552.005961,12.493506,3.592606,550.138021,0.297348,0.359261,238.315834,0.0,0.0,2.0,...,0.000375,0.894864,0.155559,1.242282,0.576698,0.024869,59.371510,10.320845,82.421626,38.262149
1,2954.069092,16.607849,6.838123,554.637586,0.170457,0.683812,240.633322,0.0,0.0,2.0,...,0.003413,8.146984,1.416231,11.309930,5.250348,0.043175,103.075538,17.918134,143.093100,66.427343
2,3491.190024,19.109881,8.081458,499.909698,0.132667,0.808146,174.137192,0.0,0.0,2.0,...,0.000558,1.331388,0.231442,1.848279,0.858017,0.045938,109.672373,19.064894,152.251059,70.678693
3,1432.773708,11.643507,3.316606,285.105331,0.156835,0.331661,114.079129,0.0,0.0,2.0,...,0.000845,2.017355,0.350687,2.800563,1.300090,0.022797,54.423884,9.460775,75.553157,35.073637
4,2708.524032,18.480233,6.269732,245.876555,0.065684,0.626973,98.168256,0.0,0.0,2.0,...,0.001331,3.177900,0.552430,4.411672,2.048007,0.031086,74.214388,12.901056,103.027032,47.827687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3671,1773.403827,14.840244,4.105101,117.194010,0.040715,0.410510,67.972264,0.0,0.0,2.0,...,0.002845,3.978386,1.429786,4.833251,3.076886,0.031961,44.700269,16.064764,54.305344,34.571211
3672,181.737818,4.892592,0.420689,118.784997,0.425960,0.042069,17.678274,0.0,0.0,2.0,...,0.000263,0.368416,0.132405,0.447581,0.284933,0.003760,5.258855,1.889972,6.388864,4.067201
3673,489.147555,5.657191,1.132286,76.796669,0.131983,0.113229,32.198044,0.0,0.0,2.0,...,0.001493,2.087616,0.750265,2.536198,1.614564,0.015511,21.692778,7.796136,26.354064,16.777205
3674,1276.439286,12.916445,2.954721,203.032799,0.117782,0.295472,58.699956,0.0,0.0,2.0,...,0.000482,0.673643,0.242100,0.818394,0.520996,0.018331,25.636919,9.213615,31.145712,19.827606


In [12]:
from utils import get_importance

imp = get_importance(X, y.drop('Cell ID unique', axis=1), sort=False, plot=True)

In [13]:
imp = imp.sort_values(ascending=False)
# display(imp)

most_imp = imp[:selection]
# most_imp = imp[imp > .005]

feat = most_imp.index

print(f'Selected features:\n{list(feat)}')
data = X[feat]

Selected features:
['Area_norm: angle from last_mov [deg] -- zc', 'AR_norm: angle from last_mov [deg] -- zc', 'Round_norm: angle from last_mov [deg] -- zc', 'Circ_norm: angle from last_mov [deg] -- zc', 'Solidity_norm: angle from last_mov [deg] -- zc', 'Area_norm: dist std', 'Area_norm: diff angle from init [deg] -- zc', 'AR_norm: diff angle from init [deg] -- zc', 'Round_norm: diff angle from init [deg] -- zc', 'AR_norm: dist std', 'Area_norm: avg speed', 'Area_norm: dist mean', 'Circ_norm: diff angle from init [deg] -- zc', 'Area_norm: total_dist', 'Area_norm: max_disp']


## Selecting best N features

In [14]:
from utils import make_cv

ver=VERBOSE

print(f'Investigate poison: {y["type"].unique()}')
for cell_line in y['cell line'].unique():
    if ver:
        print(f'\n======================================================')
    print(f'Investigation: {cell_line}')

    ind = (y['cell line'] == cell_line)
    
    make_cv(pipe, cv, data[ind].values, y[ind]['type'].values, y[ind]['Cell ID unique'], repeat=REPEAT_CV, verbose=ver)
print()

Investigate poison: ['1 uM' '2 uM' 'kontroll']
Investigation: A2058
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std
Investigation: JL Post
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std
Investigation: JR Post
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std
Investigation: JR Pre
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std
Investigation: WM983B
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std



## Selecting features - one by one

In [15]:
ver=VERBOSE

for col in X.columns:
    data = X[[col]]
    print(f'\nColumn: {col}\n\n')
    
    print(f'Investigate poison: {y["type"].unique()}')
    for cell_line in y['cell line'].unique():
        if ver:
            print(f'\n======================================================')
        print(f'Investigation: {cell_line}')
    
        ind = (y['cell line'] == cell_line)
        make_cv(pipe, cv, data[ind].values, y[ind]['type'].values, y[ind]['Cell ID unique'],repeat=REPEAT_CV,  verbose=ver)
    print()


Column: total_dist


Investigate poison: ['1 uM' '2 uM' 'kontroll']
Investigation: A2058
0.5632 +/- 0.0156 conf(0.95),	 0.0543 std
Investigation: JL Post
0.5531 +/- 0.0152 conf(0.95),	 0.0531 std
Investigation: JR Post
0.5795 +/- 0.0155 conf(0.95),	 0.0539 std
Investigation: JR Pre
0.5932 +/- 0.0129 conf(0.95),	 0.0449 std
Investigation: WM983B
0.5797 +/- 0.0133 conf(0.95),	 0.0464 std


Column: dist std


Investigate poison: ['1 uM' '2 uM' 'kontroll']
Investigation: A2058
0.5528 +/- 0.0136 conf(0.95),	 0.0475 std
Investigation: JL Post
0.5128 +/- 0.0188 conf(0.95),	 0.0653 std
Investigation: JR Post
0.5848 +/- 0.0162 conf(0.95),	 0.0563 std
Investigation: JR Pre
0.5874 +/- 0.0167 conf(0.95),	 0.0582 std
Investigation: WM983B
0.5767 +/- 0.0166 conf(0.95),	 0.0577 std


Column: dist mean


Investigate poison: ['1 uM' '2 uM' 'kontroll']
Investigation: A2058
0.5650 +/- 0.0151 conf(0.95),	 0.0527 std
Investigation: JL Post
0.5423 +/- 0.0168 conf(0.95),	 0.0586 std
Investigation: JR Post
0

## Kontroll classification

In [16]:
ver=VERBOSE
print(f'Investigate Controlls: {y["cell line"].unique()}')


ind = y['type'] == 'kontroll'

make_cv(pipe, cv, data[ind].values, y[ind]['cell line'].values, y[ind]['Cell ID unique'], repeat=REPEAT_CV, verbose=ver)

Investigate Controlls: ['A2058' 'JL Post' 'JR Post' 'JR Pre' 'WM983B']
0.8186 +/- 0.0078 conf(0.95),	 0.0273 std


In [17]:
ver=VERBOSE
print(f'Investigate Controlls: {y["cell line"].unique()} - all features')


ind = y['type'] == 'kontroll'

make_cv(pipe, cv, X[ind].values, y[ind]['cell line'].values, y[ind]['Cell ID unique'], repeat=REPEAT_CV, verbose=ver)

Investigate Controlls: ['A2058' 'JL Post' 'JR Post' 'JR Pre' 'WM983B'] - all features
1.0000 +/- 0.0000 conf(0.95),	 0.0000 std


In [18]:
# ver=VERBOSE

# # print(f'Investigation for top {selection} feature by method.')

# # print(f'Investigate poison: {y["type"].unique()}')
# for method in y['investigation'].unique():
#     feat = get_importance(X, y, plot=True)[:selection].index
#     print(f'\n{method} - Selected features:\n{list(feat)}\n')
#     # data = X[feat]
    
#     # for cell_line in y[y['investigation']==method]['cell line'].unique():
#     #     if ver:
#     #         print(f'\n======================================================')
#     #     print(f'Investigation: {method}-{cell_line}')

#     #     ind = (y['investigation'] == method) & (y['cell line'] == cell_line)
        
#     #     make_cv(pipe, cv, data[ind].values, y[ind]['type'].values, verbose=ver)